# 08. Production Concerns

**You will be able to:**

- Cut cost and latency with prompt caching
- Stream responses and handle errors like a real client
- Instrument usage so cost surprises show up early

**Prerequisites:** All previous lessons  
**Estimated API cost:** ~$0.15

---

In [ ]:
import sys
sys.path.insert(0, "../src")

from genai_tutorial import ask, get_client, stream_text, usage_line, DEFAULT_MODEL

client = get_client()
print("Using model:", DEFAULT_MODEL)

## 1. Prompt caching

Caching is a **prefix match**. Any byte change anywhere in the prefix invalidates everything after it. Render order is `tools` → `system` → `messages`, so stable content goes first and volatile content (timestamps, the user's question) goes last.

In [ ]:
LARGE_CONTEXT = open("../data/raw/sample.md", encoding="utf-8").read()

for i in range(2):
    response = client.messages.create(
        model=DEFAULT_MODEL,
        max_tokens=1000,
        system=[{
            "type": "text",
            "text": LARGE_CONTEXT,
            "cache_control": {"type": "ephemeral"},
        }],
        messages=[{"role": "user", "content": "Summarise this in one line."}],
    )
    print(f"call {i}:", usage_line(response))

On the second call `cache_read` should jump and `in` should collapse. If `cache_read` stays at 0, something in the prefix is changing between calls — a timestamp, an unsorted `json.dumps`, a reordered tool list. That is the first thing to check, always.

Note the minimum cacheable prefix is around 1024 tokens; shorter prefixes silently will not cache.

## 2. Streaming

Stream anything a human waits on. It also sidesteps HTTP timeouts on large `max_tokens`.

In [ ]:
for chunk in stream_text("Explain prompt caching to a sceptical CFO."):
    print(chunk, end="", flush=True)

## 3. Errors

Catch a chain, not one broad class — retryable (429, 5xx, connection) and non-retryable (400, 404) failures need different responses.

In [ ]:
import anthropic

try:
    ask("hello", model="claude-does-not-exist")
except anthropic.NotFoundError as e:
    print("bad model:", e.message)
except anthropic.RateLimitError as e:
    print("back off for", e.response.headers.get("retry-after"))
except anthropic.APIConnectionError:
    print("network problem")

## Exercise 08.1

Instrument `ask()` to log model, latency, and token usage per call. Run lesson 07's eval suite through it and produce a cost-per-eval-run figure. That number is what tells you whether nightly evals are affordable.

In [ ]:
# Your code here

---

**Next:** see `README.md` for the full syllabus. Stuck? `docs/troubleshooting.md`. Solutions live in `solutions/` — try the exercise first.